In [ ]:
# Installing Libraries
!pip install category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 3.4 MB/s eta 0:00:00


In [ ]:
# Importing Libraries
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import FunctionTransformer, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

from category_encoders import TargetEncoder

In [ ]:
# Load Dataset
df = pd.read_csv("/content/drive/MyDrive/GitHub/4_Machine_Learning/3. Supervised Learning/1_Linear_Regression/Cleaned_Delhi_House_Data.csv")

In [ ]:
#Spliting Dataset
X = df.drop('Price', axis=1)
y = np.log1p(df['Price'])

In [ ]:
# Defining columns
numeric_feature = ['Area', 'Per_Sqft']
categorical_feature = ['Furnishing', 'Status', 'Transaction', 'Type']
target_feature = ['Locality']

In [ ]:
# Pipeline for missing values imputation
# Pipeline for Mean Imputation and Log Transformation for Numeric feature
numeric_pipeline = Pipeline([
    ('log_transform', FunctionTransformer(np.log1p, validate=False))
])

# Pipeline for Mode Imputation and Ordinal Encoding for Catergorical feature
categorical_pipeline = Pipeline([
    ('ord_encoder', OrdinalEncoder())
])

# Pipeline for Target Encoding
target_pipeline = Pipeline([
    ('tar_encoder', TargetEncoder())
])

In [ ]:
# Combine all pipeline as a preprocessor
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_feature),
    ('cat', categorical_pipeline, categorical_feature),
    ('target', target_pipeline, target_feature)
    ], remainder='passthrough')

In [ ]:
# Final Pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

In [ ]:
# Spliting training and testing dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Model Training
pipeline.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num',
                                                  Pipeline(steps=[('log_transform',
                                                                   FunctionTransformer(func=<ufunc 'log1p'>))]),
                                                  ['Area', 'Per_Sqft']),
                                                 ('cat',
                                                  Pipeline(steps=[('ord_encoder',
                                                                   OrdinalEncoder())]),
                                                  ['Furnishing', 'Status',
                                                   'Transaction', 'Type']),
                                                 ('target',
                                                  Pipeline(steps=[('tar_encoder',
                                                                   TargetEncoder())]),
                                                  ['Locality'])])),
                ('model', LinearRegression())])

In [11]:
y_pred = pipeline.predict(X_test)

In [12]:
mae = mean_absolute_error(y_test, y_pred)

mse = mean_squared_error(y_test, y_pred)

rmse = np.sqrt(mse)

r2 = r2_score(y_test, y_pred)

In [13]:
print("Linear Regression Performance")

print(f"MAE  : {mae:.2f}")

print(f"MSE  : {mse:.2f}")

print(f"RMSE : {rmse:.2f}")

print(f"R² Score : {r2:.4f}")

Linear Regression Performance
MAE  : 0.34
MSE  : 0.19
RMSE : 0.44
R² Score : 0.8279


In [14]:
# Comparision
comparison = pd.DataFrame({
    'Actual Price': y_test,
    'Predicted Price': y_pred
})

comparison.head(10)

,Actual Price,Predicted Price
76,16.516872,16.127266
1026,17.054189,17.216129
43,15.869634,16.029443
666,17.776324,17.662205
529,14.483340,14.735551
101,17.370859,17.480519
908,17.776324,17.742730
1224,18.515991,17.933032
777,15.319588,15.359235
453,16.733281,16.622890


In [15]:
# Saving the model
import joblib

In [16]:
# Save the entire pipeline (preprocessing + model)
joblib.dump(pipeline, "House_price_prediction_pipeline.pkl")
print("Model saved as House_price_prediction_pipeline.pkl")

Model saved as House_price_prediction_pipeline.pkl


In [17]:
# Load the pipeline
loaded_model = joblib.load("/content/drive/MyDrive/GitHub/4_Machine_Learning/3. Supervised Learning/1_Linear_Regression/House_price_prediction_pipeline.pkl")

In [19]:
sample = X_test.sample(1)
sample

,Area,BHK,Bathroom,Furnishing,Locality,Parking,Status,Transaction,Type,Per_Sqft
1255,1050.0,3,2.0,Semi-Furnished,Chittaranjan Park,3.0,Ready_to_move,Resale,Builder_Floor,12916.0


In [20]:
load_pred = loaded_model.predict(sample)
load_pred

array([16.54943462])